# Lesson 7, Part 1

Let's code some basic operations for dealing with complex numbers.

In [ ]:
import numpy as np

def cx(x,y): return (x,y)  # form complex number x+iy as a tuple

def real(z): return z[0]   # real component of z

def imag(z): return z[1]   # imaginary component of z

def conjugate(z):         # conjugate of z
    return (real(z),-imag(z))

def magnitude(z):          # magnitude of z
    return (real(z)**2 + imag(z)**2)**0.5

def cx_add(w,z):          # add two complex numbers
    return (real(w) + real(z),imag(w) + imag(z))

def cx_subtract(w,z):     # subtract one complex number from another
    return (real(w) - real(z),imag(w) - imag(z))

def cx_mult(w,z):         # multiply two complex numbers
    real_mult = real(w) * real(z) - imag(w) * imag(z)
    imag_mult = real(w) * imag(z) + imag(w) * real(z)
    return (real_mult,imag_mult)

def cx_divide(w,z):       # divide two complex numbers
    numer = cx_mult(w, conjugate(z))
    denom = real(z)**2 + imag(z)**2
    return (real(numer)/denom, imag(numer)/denom)

w,z = cx(1,5),cx(3,4)
print('w =',w,'has real component',real(w),'and imaginary component',imag(w))
print('z =',z,'has real component',real(z),'and imaginary component',imag(z))
print('z has conjugate',conjugate(z),'and magnitude',magnitude(z))
v = cx_add(w,z)
print('w + z =',v,'with real component',real(v),'and imaginary component',imag(v))
u = cx_subtract(w,z)
print('w - z =',u,'with real component',real(u),'and imaginary component',imag(u))
v = cx_mult(w,z)
print('v = w * z =',v,'with real component',real(v),'and imaginary component',imag(v))
u = cx_divide(v,z)
print('u = v / z =',u,', which is',cx_subtract(u,w),'different from w')

Fortunately, Python builds in simpler ways to call all these functions.

In [ ]:
w = complex(1,5)
w1 = 1+5j
print("Are complex(1,5) and 1+5j identical in Python?",w==w1)
print("What type does this represent?",type(w))
z = 3+4j
real_z = z.real
imag_z = z.imag
print('z =',z,'has real component',real_z,'and imaginary component',imag_z)
conj_z = z.conjugate()
magnitude_z = abs(z)
print('Conjugate value of z is',conj_z,'with magnitude',magnitude_z)
v = w + z
u = w - z
print('w + z =',v,'and w - z =',u)
v = w * z
u = v / z
print('w * z =',w*z,' and w / z =',w/z)
print('Difference  between  (w * z) / z and w is', (w * z) / z - w)

Numpy builds in versions of these functions that work with ndarrays. 
It also has a function **real_if_close** that rounds tiny imaginary components to zero.

In [ ]:
for q in 2+5j,3+4j:
    print('q =',q,'has real component q.real =',q.real,'or np.real(q) =',np.real(q))
    print('Imaginary component is q.imag =',q.imag,'or np.imag(q) =',np.imag(q))
    print('\nq.conjugate =',q.conjugate(),'or np.conj(q) = np.conjugate(q) =',np.conj(q))
    print('Magnitude np.abs(q) =',np.abs(q))
    print('\n2 * q =',2*q,',  1  / q =',1/q)
    v = q * (1/q)
    print('\nq * (1 / q) =',v,',which rounds to',np.real_if_close(v),'\n')

As we have seen, complex numbers can also be expressed in the form *z* = *r* exp(*i*$\theta$) or the polar shorthand (*r*,$\theta$). 
Here are Taylor series that convert them into rectangular *x*+*iy* format, with *x* = *r* cos $\theta$ and *y* = *r* sin $\theta$.

In [ ]:
def taylor_cos(theta,countOrder=False,tol=1E-15):
    theta_square = theta * theta
    cosine = next_term = 1
    k = 0
    while abs(next_term) > tol:
        k += 2
        next_term *= -theta_square / (k * (k-1))
        cosine += next_term
    return (cosine,k) if countOrder else cosine

def taylor_sin(theta,countOrder=False,tol=1E-15):
    theta_square = theta * theta
    sine = next_term = theta
    k = 1
    while abs(next_term) > tol:
        k += 2    
        next_term *= -theta_square / (k * (k-1))
        sine += next_term
    return (sine,k) if countOrder else sine

for m in np.linspace(-1,1,9):
    cosine = taylor_cos(m * np.pi)
    sine = taylor_sin(m * np.pi)
    print('cos (',m,'π) =','{:.4f}'.format(cosine),', sin(',m,'π) =','{:.4f}'.format(sine))

For very large *theta*, the previous code fails to converge 
before rounding errors overwhelm the results.

In [ ]:
print('Multiples of 2π where true cosine = 1 and true sine = 0:')
for m in (4,8,16,32):
    cosine,k_cos = taylor_cos(m * np.pi, True)
    sine,k_sin = taylor_sin(m * np.pi, True)
    print('cos (',m,'π) =','{:.1e}'.format(cosine),', sin(',m,'π) =','{:.1e}'.format(sine), \
          'with orders',k_cos,'for cosine and',k_sin,'for sine')

Shifting *theta* by multiples of 2π can fix this.

In [ ]:
def shift_angle(theta):  
    theta = theta % (2 * np.pi)
    return theta if theta <= np.pi else theta - 2 * np.pi
    # previous line equivalent to lines below
    # if theta > np.pi: theta -= 2 * np.pi
    # return theta    

for theta in np.geomspace(1E1,1E6,6):
    new_theta = shift_angle(theta)
    print('θ =','{:7.0f}'.format(theta),'shifted to','{:.6f}'.format(new_theta))
    
print('\nOrder needed for convergence of taylor series with -pi <= theta <= pi')
for theta in np.linspace(-np.pi,np.pi,21):
    cosine,k_cos = taylor_cos(theta,True)
    sine,k_sin = taylor_sin(theta,True)
    print('θ =',round(theta,3),': cosine',k_cos,', sine',k_sin)

This allows us to convert polar notation to rectangular *x*+*iy* form in close approximation to what **np.sin** and **np.cos** can do. 
The *r* of the associated polar circle is commonly known as the modulus.

In [ ]:
def rectangular(modulus,theta):
    theta = shift_angle(theta)
    x = modulus * taylor_cos(theta)
    y = modulus * taylor_sin(theta)
    return x + y * 1j

modulus = 1
print('Difference between taylor series and built-in functions')
for theta in (0.5,1,2,3,30,60,-60,-30):
    z = rectangular(modulus,theta)
    z1 = modulus * (np.cos(theta) + np.sin(theta) * 1j)
    print('θ =',theta,':','{:.1e}'.format(z-z1),'with magnitude','{:.0e}'.format(abs(z-z1)))

Let's use rectangular( ) to compute complex roots of 1.

In [ ]:
def roots_of_1(power,tol=2E-14):   
    roots = []
    curr_root = 1
    base_root = rectangular(1,2*np.pi/power)
    for i in range(power):
        roots.append(curr_root)
        curr_root *= base_root
    diverge = abs(curr_root - 1)
    if diverge > tol:
        print('Warning: divergence of','{:.1e}'.format(diverge),'for power',power)
    return(roots)
        
print('Roots of 1 (other than 1)')
for power in range(2,10):
    roots = roots_of_1(power)[1:]
    for i in range(power-1):
        roots[i] = round(roots[i].real,5) + round(roots[i].imag,5)*1j
    print('power',power,':',roots)

**np.exp** handles complex numbers too and is faster than calling **np.cos** and **np.sin** separately. 
If we evaluate only a few numbers, **math.cos** and **math.sin** are much faster but 

In [ ]:
import math
from time import time as T

modulus,theta = 3,20
iters,tic = 10000,T()
for iter in range(iters): z = rectangular(modulus,theta)
print(z,'\nrectangular() takes time',T()-tic); tic = T()
for iter in range(iters): z = modulus * (np.cos(theta) + np.sin(theta) * 1j) 
print(z,'\nnp.cos() and np.sin() take time',T()-tic); tic = T()
for iter in range(iters): modulus * np.exp(theta * 1j)
print(z,'\nnp.exp() takes time',T()-tic); tic = T()
for iter in range(iters): modulus * (math.cos(theta) + math.sin(theta) * 1j)
print(z,'\nmath.sin() and math.cos() take time',T()-tic);

Next let's convert complex numbers from rectangular format to polar format, 
using a Newton-Raphson method to identify the inverse cosine.

In [ ]:
def inv_cos(x,countOrder=False,tol=1E-15):
    theta = np.pi/2
    diverge = -x
    k = 0
    while abs(diverge) > tol:
        k += 1
        theta += diverge / taylor_sin(theta)
        diverge = taylor_cos(theta) - x
    return (theta,k) if countOrder else theta

def polar_coord(z):
    r = abs(z)
    theta = abs(inv_cos(z.real/r))
    if z.imag < 0:
        theta = - theta
    return r,theta
    
# test
modulus = 0
print('Errors in conversion of polar format to rectangular and back to polar')
for theta in np.linspace(-3,3,13):
    modulus += 0.5
    z = rectangular(modulus,theta)
    r,thet = polar_coord(z)
    diverge_r = r - modulus
    diverge_thet = thet - theta
    print('r = ',modulus,': error','{:.1e}'.format(diverge_r),' θ =',theta,': error','{:.1e}'.format(diverge_thet))

The inv_cos algorithm converges in 3-6 iterations 
except when cos(*theta*) is very close to 1 or -1 and hence very flat.

In [ ]:
print('Iterations to Convergence')
for x in np.linspace(-1,1,11):
    theta,k = inv_cos(x,True)
    print('x =',round(x,1),':',k)
print('Behavior near 1')    
for gap in np.geomspace(0.1,1E-9,9):    
    theta,k = inv_cos(1-gap,True)
    print('x =',1-gap,':',k)

Numpy offers two ways to improve on polar(). The first elicits the angle with **angle**. 
The second uses **log** to elicit ln(modulus) as the real component and the angle as the imaginary component.

In [ ]:
isequal_angle = True
max_diff_r0 = max_diff_r1 = max_diff_angle = 0
for x in np.linspace(-10,10,20):
    for y in np.linspace(-10,10,20):
        z = x + y * 1j
        r,theta = polar_coord(z)
        logz = np.log(z)
        # update cumulative maximum
        diff0 = abs(np.abs(z) - r)
        if diff0 > max_diff_r0:
            max_diff_r0 = diff0 
        # another way to update cumulative maximum
        diff1 = abs(np.log(r) - logz.real)
        max_diff_r1 = max(diff1, max_diff_r1)
        # another way to update cumulative maximum
        max_diff_angle = max(abs(np.angle(z) - theta), max_diff_angle)
        
#print('Does modulus r of polar(z) match np.abs(z)?',isequal_r)
print('Maximum difference in log modulus between log(r) and np.abs(z) is','{:.1e}'.format(max_diff_r0))
print('Maximum difference in log modulus between log(r) and np.log(z).real is','{:.1e}'.format(max_diff_r1))
print('\nDoes np.angle(z) match np.log(z).imag?',isequal_angle)
print('Maximum difference in angle between polar(z) and np.angle(z) is','{:.1e}'.format(max_diff_angle))

There is also a separate package cmath for complex math that offers a fast complex **exp**, 
a better version of rectangular called **rect**, and a better version of polar_coord called **polar**. 
The angle alone can be elicited with **phase**.

In [ ]:
import numpy as np
import cmath
from time import time as T

isequal = True
for r in np.linspace(1,10,20):
    for theta in np.linspace(-np.pi,np.pi,20):
        if not(cmath.rect(r,theta) == r * np.exp(theta*1j) ==  r * cmath.exp(theta*1j)):
            isequal = False

print('Does cmath.rect(r,θ), r * np.exp(iθ), and r * cmath.exp(iθ) match?',isequal)
modulus,theta  = 3,2
iters,tic = 10000,T()
for iter in range(iters): modulus * np.exp(theta * 1j)
print('modulus * np.exp(theta * 1j) takes time',T()-tic); tic = T();
for iter in range(iters): modulus * cmath.exp(theta * 1j)
print('modulus * cmath.exp(theta * 1j) takes time',T()-tic); tic = T();
for iter in range(iters): cmath.rect(modulus,theta)
print('cmath.rect(modulus,theta) takes time',T()-tic); 

max_diff_r,isequal_angle = 0,True
for x in np.linspace(-10,10,20):
    for y in np.linspace(-10,10,20):
        z = x + y * 1j
        diff0 = abs(cmath.polar(z)[0] - np.abs(z))
        if diff0 > max_diff_r:
            max_diff_r = diff0
        if not(cmath.polar(z)[1] == cmath.phase(z) == np.angle(z)):
            isequal_angle = False

print('\nMaximum difference between c.polar[0] and np.abs is',max_diff_r)
print('Do cmath.polar[1], cmath.phase, and np.angle match?',isequal_angle)
tic = T()
for iter in range(iters): np.angle(z)
print('np.angle() takes time',T()-tic); tic = T()
for iter in range(iters): cmath.polar(z)
print('cmath.polar() takes time',T()-tic); tic = T()
for iter in range(iters): cmath.phase(z)
print('cmath.phase() takes time',T()-tic)